# 5. Reranking en dos etapas: ¿dónde se pierde la información?

**Proyecto 2 MCC225** · Metodología del Cuaderno 6 (Semana 3) adaptada al dominio

## El problema que arrastran los notebooks anteriores

Tres resultados que hasta ahora no se explicaban entre sí:

1. Los captions largos **no superan** a los cortos (notebooks 2 y 3).
2. Al quitar el título impreso, el R@1 cae **a la mitad** (0.684 → 0.326).
3. En la prueba composicional los tres modelos quedan **en el azar** (notebook 4).

La hipótesis que los une es arquitectónica. CLIP es un **bi-encoder**: comprime
la imagen entera en un solo vector *antes* de ver el texto. La posición relativa
de dos barras, la dirección de una serie o el valor de un eje son cosas que
existen en los parches de la imagen pero que el vector único no conserva.

Si esa hipótesis es correcta, la información no está mal codificada: está
**perdida en el pooling**. Y entonces ningún caption, por largo que sea, podría
aprovecharla.

## Cómo lo prueba este notebook

Se entrenan dos rerankers sobre los **mismos** embeddings congelados de CLIP-L,
que se diferencian únicamente en qué recibe cada uno:

| Reranker | Entrada de imagen | Qué puede hacer |
|---|---|---|
| **Pooled** | el vector único (después del pooling) | combinar imagen y texto, corregir calibración |
| **Patches** | los 256 vectores de parche (antes del pooling) | atender a regiones guiado por el texto |

Ambos reordenan el top-K que recuperó el bi-encoder, como en el Cuaderno 6. La
comparación entre los dos es el experimento:

- Si **solo el de parches** mejora, la información existe en la imagen y el
  pooling la destruye. La limitación es del bi-encoder, no de los datos.
- Si **ninguno** mejora, la información no está codificada en absoluto.
- Si **ambos** mejoran por igual, lo que faltaba era calibración, no contenido.

## Restricción de datos y cómo se maneja

El Cuaderno 6 entrena con 800 imágenes de Flickr8k. Aquí hay 95, así que un
cross-encoder completo se sobreajustaría. Dos decisiones lo compensan:

- **CLIP queda congelado.** Solo se entrenan las cabezas de reranking, que tienen
  pocos parámetros.
- **Validación dejando una edición fuera.** Se entrena con dos ediciones y se
  evalúa en la tercera, nunca vista. Con tres ediciones salen tres pliegues, y la
  estructura multiedición pasa de ser un dato más a ser el mecanismo de control.

Se usan las imágenes **sin título** porque son la medición limpia: con el título,
cualquier mejora podría venir de leer texto impreso.

In [1]:
from pathlib import Path
import json, random, sys, platform

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image

BASE = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
MULTI = BASE / "multiedicion"
# --- variante de imágenes -------------------------------------------------
# Un único punto decide si se trabaja con las imágenes que llevan el título
# impreso o con las recortadas. El manifiesto es el mismo para ambas; cambia
# solo la subcarpeta. El nombre del directorio de salida arrastra la variante,
# de modo que dos corridas nunca se sobrescriben.
import sys
sys.path.insert(0, str(BASE))
from config_variante import (VARIANTE, carpeta_imagenes, ruta_imagen,
                             dir_salida, resumen)

IMGS = carpeta_imagenes(MULTI)
print(resumen(MULTI))

OUT = dir_salida(BASE, "resultados_reranking")

SEED = 22514
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

CHECKPOINT = "openai/clip-vit-large-patch14"   # el mejor en los notebooks previos
# la variante se elige en config_variante.py
CAPTION = "caption_3"                          # el que más información aporta
TOPK = 10                                      # como en el Cuaderno 6
EPOCHS = 20               # pocas épocas: el sobreajuste llega rápido
LR = 1e-3

man = pd.read_csv(MULTI / "manifest_multiedicion.csv")
man = man[man.chart_uid != "2024-2|III.A.1"]   # recorte sin título no válido
print(f"{len(man)} pares")
print(man.groupby("edicion").size().to_string())
print(f"\ndispositivo: {DEVICE}")

variante: con_titulo | carpeta: con_titulo | 95 imágenes
94 pares
edicion
2021-1    26
2024-2    28
2026-1    40

dispositivo: cuda


## 5.1 Extracción de rasgos congelados

De cada imagen se guardan dos representaciones:

- **`emb_img`**: el vector final proyectado, que es lo que usa CLIP para el
  retrieval habitual.
- **`patches`**: los estados ocultos del encoder visual antes del pooling, es
  decir 256 vectores de parche (ViT-L/14 sobre 224×224).

La segunda es la que contiene la estructura espacial. Extraerlas juntas garantiza
que ambos rerankers vean exactamente la misma imagen procesada igual.

In [2]:
from transformers import CLIPModel, CLIPProcessor

modelo = CLIPModel.from_pretrained(CHECKPOINT, use_safetensors=True).to(DEVICE).eval()
proc = CLIPProcessor.from_pretrained(CHECKPOINT)

@torch.no_grad()
def extraer(rutas, textos):
    embs_i, patches, embs_t = [], [], []

    for r in rutas:
        px = proc(images=Image.open(r).convert("RGB"),
                  return_tensors="pt")["pixel_values"].to(DEVICE)
        vis = modelo.vision_model(pixel_values=px)
        # last_hidden_state: [1, 1+n_parches, ancho]; se descarta el token CLS
        patches.append(vis.last_hidden_state[:, 1:, :].squeeze(0).cpu())
        e = modelo.visual_projection(modelo.vision_model.post_layernorm(vis.pooler_output))
        embs_i.append(F.normalize(e, dim=-1).squeeze(0).cpu())

    for t in textos:
        inp = proc(text=[t], return_tensors="pt", padding=True,
                   truncation=True, max_length=77).to(DEVICE)
        e = modelo.get_text_features(**inp)
        embs_t.append(F.normalize(e, dim=-1).squeeze(0).cpu())

    return torch.stack(embs_i), torch.stack(patches), torch.stack(embs_t)


rutas = [ruta_imagen(MULTI, p) for p in man["image_path"]]
E_img, P_img, E_txt = extraer(rutas, man[CAPTION].astype(str).tolist())

print(f"embeddings de imagen: {tuple(E_img.shape)}")
print(f"parches por imagen:   {tuple(P_img.shape)}")
print(f"embeddings de texto:  {tuple(E_txt.shape)}")
print(f"\nel vector único resume {P_img.shape[1]} parches en {E_img.shape[1]} dimensiones")

/tf/work/torch_gpu_env/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


embeddings de imagen: (94, 768)
parches por imagen:   (94, 256, 1024)
embeddings de texto:  (94, 768)

el vector único resume 256 parches en 768 dimensiones


## 5.2 Etapa 1: recuperación con el bi-encoder

El punto de partida. Se calculan las similitudes dentro de cada edición —pools
separados, como en el notebook 3— y se guarda el top-K de cada imagen.

In [3]:
def rangos(sim):
    orden = np.argsort(-sim, axis=1)
    return np.array([int(np.where(orden[i] == i)[0][0]) + 1 for i in range(len(sim))])


idx_por_edicion, base_metricas = {}, []
for ed, g in man.groupby("edicion"):
    pos = np.where(man["edicion"].values == ed)[0]
    idx_por_edicion[ed] = pos
    sim = (E_img[pos] @ E_txt[pos].T).numpy()
    r = rangos(sim)
    base_metricas.append({"edicion": ed, "n": len(pos),
                          "R@1": round(float((r == 1).mean()), 4),
                          "R@5": round(float((r <= 5).mean()), 4),
                          "MRR": round(float((1 / r).mean()), 4)})

base = pd.DataFrame(base_metricas)
print("Etapa 1 — bi-encoder CLIP-L (imágenes sin título)")
print(base.to_string(index=False))

Etapa 1 — bi-encoder CLIP-L (imágenes sin título)
edicion  n    R@1    R@5    MRR
 2021-1 26 0.5769 0.9231 0.7399
 2024-2 28 0.6786 0.8571 0.7498
 2026-1 40 0.6500 0.8500 0.7487


## 5.3 Minado de negativos semi-duros

El Cuaderno 6 usa `negatives_for_reranker = "semihard"`. Un negativo semi-duro es
un candidato que el bi-encoder colocó arriba pero que es incorrecto: son los casos
que el reranker debe aprender a separar.

Elegirlos al azar entrenaría con pares obviamente distintos y no enseñaría nada
útil; el modelo aprendería a distinguir temas, que es lo que ya sabe hacer.

In [4]:
def minar(pos, topk=TOPK):
    """Devuelve (indice_imagen, indice_texto, etiqueta) dentro de una edición."""
    sim = (E_img[pos] @ E_txt[pos].T).numpy()
    orden = np.argsort(-sim, axis=1)
    ejemplos = []
    for i in range(len(pos)):
        ejemplos.append((pos[i], pos[i], 1.0))                    # positivo
        duros = [j for j in orden[i][:topk] if j != i]            # semi-duros
        for j in duros:
            ejemplos.append((pos[i], pos[j], 0.0))
    return ejemplos


todos = {ed: minar(p) for ed, p in idx_por_edicion.items()}
for ed, ej in todos.items():
    pos_n = sum(1 for e in ej if e[2] == 1)
    print(f"{ed}: {len(ej)} ejemplos ({pos_n} positivos, {len(ej)-pos_n} negativos semi-duros)")

2021-1: 261 ejemplos (26 positivos, 235 negativos semi-duros)
2024-2: 282 ejemplos (28 positivos, 254 negativos semi-duros)
2026-1: 402 ejemplos (40 positivos, 362 negativos semi-duros)


## 5.4 Las dos cabezas de reranking

**Pooled.** Recibe los dos vectores y sus interacciones clásicas —producto
elemento a elemento y diferencia absoluta— que es la forma habitual de comparar
representaciones. No puede acceder a la estructura espacial porque el vector ya
no la contiene.

**Patches.** Usa el vector de texto como consulta de atención sobre los 256
parches: para cada par, calcula qué regiones de la imagen son relevantes para ese
texto y las agrega ponderadamente. Es atención cruzada, la operación que un
bi-encoder no puede hacer.

Ambas terminan en un logit binario de correspondencia, como el reranker del
Cuaderno 6.

In [5]:
class RerankerPooled(nn.Module):
    def __init__(self, dim, oculto=64):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(dim * 4, oculto), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(oculto, 1))

    def forward(self, ei, et, patches=None):
        x = torch.cat([ei, et, ei * et, (ei - et).abs()], dim=-1)
        return self.mlp(x).squeeze(-1)


class RerankerPatches(nn.Module):
    """El texto consulta los parches de la imagen mediante atención."""
    def __init__(self, dim_txt, dim_patch, oculto=64):
        super().__init__()
        self.q = nn.Linear(dim_txt, oculto)
        self.k = nn.Linear(dim_patch, oculto)
        self.v = nn.Linear(dim_patch, oculto)
        self.mlp = nn.Sequential(
            nn.Linear(oculto + dim_txt, oculto), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(oculto, 1))
        self.escala = oculto ** 0.5

    def forward(self, ei, et, patches):
        q = self.q(et).unsqueeze(1)                    # [B, 1, oculto]
        k, v = self.k(patches), self.v(patches)        # [B, n_parches, oculto]
        att = torch.softmax((q @ k.transpose(1, 2)) / self.escala, dim=-1)
        ctx = (att @ v).squeeze(1)                     # regiones relevantes
        return self.mlp(torch.cat([ctx, et], dim=-1)).squeeze(-1)


# Las cabezas se mantienen deliberadamente pequeñas: con ~69 imágenes por
# pliegue, una capa oculta de 256 unidades daría cerca de un millón de
# parámetros y memorizaría el conjunto de entrenamiento. Con 64 unidades y
# weight decay el número baja un orden de magnitud.
for nombre, m in [("Pooled ", RerankerPooled(E_img.shape[1])),
                  ("Patches", RerankerPatches(E_img.shape[1], P_img.shape[2]))]:
    print(f"{nombre}: {sum(p.numel() for p in m.parameters()):,} parámetros")
print("\nPooled  — ve el vector único, no puede mirar regiones")
print("Patches — atiende sobre los parches guiado por el texto")

Pooled : 196,737 parámetros
Patches: 233,793 parámetros

Pooled  — ve el vector único, no puede mirar regiones
Patches — atiende sobre los parches guiado por el texto


## 5.5 Entrenamiento dejando una edición fuera

Cada pliegue entrena con dos ediciones y evalúa en la tercera, que el modelo no
vio nunca. Es una división por grupos, no aleatoria: separar al azar los 95 pares
dejaría gráficos de la misma edición a ambos lados y el resultado estaría
contaminado.

Es exactamente el problema que el examen plantea en la Tarea B del banco común
sobre divisiones inválidas cuando varias filas pertenecen al mismo grupo.

In [6]:
def entrenar_y_evaluar(clase, nombre, **kwargs):
    filas = []
    for ed_test in man["edicion"].unique():
        ejs = [e for ed, lista in todos.items() if ed != ed_test for e in lista]
        ii = torch.tensor([e[0] for e in ejs])
        jj = torch.tensor([e[1] for e in ejs])
        yy = torch.tensor([e[2] for e in ejs])

        torch.manual_seed(SEED)
        modelo_rr = clase(**kwargs).to(DEVICE)
        opt = torch.optim.Adam(modelo_rr.parameters(), lr=LR,
                               weight_decay=1e-2)   # regularización fuerte: pocos datos

        ei, et = E_img[ii].to(DEVICE), E_txt[jj].to(DEVICE)
        pt = P_img[ii].to(DEVICE)
        y = yy.to(DEVICE)
        # los positivos son 1 de cada TOPK: se compensa el desbalance
        peso = torch.where(y == 1, float(TOPK - 1), 1.0)

        modelo_rr.train()
        for _ in range(EPOCHS):
            opt.zero_grad()
            logit = modelo_rr(ei, et, pt)
            loss = F.binary_cross_entropy_with_logits(logit, y, weight=peso)
            loss.backward(); opt.step()

        # evaluación en la edición retenida: reordenar el top-K
        pos = idx_por_edicion[ed_test]
        sim = (E_img[pos] @ E_txt[pos].T).numpy()
        orden = np.argsort(-sim, axis=1)
        modelo_rr.eval()
        nuevos = []
        with torch.no_grad():
            for i in range(len(pos)):
                cand = orden[i][:TOPK]
                ei_b = E_img[pos[i]].unsqueeze(0).repeat(len(cand), 1).to(DEVICE)
                pt_b = P_img[pos[i]].unsqueeze(0).repeat(len(cand), 1, 1).to(DEVICE)
                et_b = E_txt[pos[cand]].to(DEVICE)
                s = modelo_rr(ei_b, et_b, pt_b).cpu().numpy()
                reord = cand[np.argsort(-s)]
                nuevos.append(int(np.where(reord == i)[0][0]) + 1
                              if i in reord else TOPK + 1)

        r = np.array(nuevos)
        filas.append({"reranker": nombre, "edicion_test": ed_test, "n": len(pos),
                      "R@1": round(float((r == 1).mean()), 4),
                      "R@5": round(float((r <= 5).mean()), 4),
                      "MRR": round(float((1 / r).mean()), 4)})
        print(f"  {nombre} | test={ed_test} (n={len(pos)}): R@1={filas[-1]['R@1']:.3f}")
    return filas


dim = E_img.shape[1]
dim_patch = P_img.shape[2]

print("Reranker Pooled:")
res_pooled = entrenar_y_evaluar(RerankerPooled, "pooled", dim=dim)
print("\nReranker Patches:")
res_patch = entrenar_y_evaluar(RerankerPatches, "patches",
                               dim_txt=dim, dim_patch=dim_patch)

Reranker Pooled:
  pooled | test=2021-1 (n=26): R@1=0.231
  pooled | test=2024-2 (n=28): R@1=0.107
  pooled | test=2026-1 (n=40): R@1=0.275

Reranker Patches:
  patches | test=2021-1 (n=26): R@1=0.115
  patches | test=2024-2 (n=28): R@1=0.036
  patches | test=2026-1 (n=40): R@1=0.175


## 5.6 Resultado

In [7]:
rr = pd.DataFrame(res_pooled + res_patch)
base_r = base.rename(columns={"edicion": "edicion_test"}).assign(reranker="bi-encoder")
tabla = pd.concat([base_r, rr], ignore_index=True)
tabla.to_csv(OUT / "reranking_por_edicion.csv", index=False)

resumen = (tabla.groupby("reranker")
                .apply(lambda g: pd.Series({
                    "R@1": round(float(np.average(g["R@1"], weights=g.n)), 4),
                    "R@5": round(float(np.average(g["R@5"], weights=g.n)), 4),
                    "MRR": round(float(np.average(g["MRR"], weights=g.n)), 4),
                }))
                .reindex(["bi-encoder", "pooled", "patches"]))
resumen.to_csv(OUT / "reranking_resumen.csv")
print(resumen.to_string())

b = resumen.loc["bi-encoder", "R@1"]
print(f"\nmejora del reranker pooled:  {resumen.loc['pooled','R@1'] - b:+.4f}")
print(f"mejora del reranker patches: {resumen.loc['patches','R@1'] - b:+.4f}")

               R@1     R@5     MRR
reranker                          
bi-encoder  0.6383  0.8723  0.7466
pooled      0.2128  0.6383  0.4129
patches     0.1170  0.6276  0.3401

mejora del reranker pooled:  -0.4255
mejora del reranker patches: -0.5213


## 5.7 Cómo leer el resultado

| Patrón observado | Qué significa |
|---|---|
| Solo **patches** mejora | La información espacial existe en la imagen y el pooling la destruye. La limitación es del bi-encoder, y los captions largos podrían servir con otra arquitectura |
| **Ambos** mejoran parecido | Lo que faltaba era calibración de las similitudes, no contenido. Coherente con la hubness observada en el notebook 4 |
| **Ninguno** mejora | La información no está codificada. Ni siquiera los parches de CLIP la retienen, y el problema es del encoder visual, no del pooling |
| Alguno **empeora** | Sobreajuste. Con ~69 ejemplos de entrenamiento por pliegue es un desenlace posible y debe reportarse |

**Limitaciones que hay que declarar.** El entrenamiento usa ~69 imágenes por
pliegue, muy poco para un cross-encoder aunque las cabezas sean pequeñas. La
variación entre pliegues es alta y conviene mirarla antes que el promedio. Y
CLIP permanece congelado: esto mide qué información sobrevive en sus rasgos, no
qué lograría un modelo entrenado para el dominio.

**Lo que sí aporta.** Es la única medición del proyecto que separa *información
ausente* de *información perdida en el pooling*, que es la pregunta que quedó
abierta al ver que los captions largos no aportaban nada.

In [8]:
config = {
    "experimento": "reranking_dos_etapas",
    "metodologia": "Cuaderno 6, Semana 3 (MCC225), adaptado a datos pequeños",
    "bi_encoder": CHECKPOINT,
    "variante_imagenes": VARIANTE,
    "caption": CAPTION,
    "topk": TOPK,
    "negativos": "semi-duros del top-K del bi-encoder",
    "validacion": "dejando una edición fuera (3 pliegues)",
    "clip_congelado": True,
    "epochs": EPOCHS, "lr": LR, "seed": SEED,
    "n_total": int(len(man)),
    "runtime": {"python": sys.version.split()[0], "torch": torch.__version__,
                "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None},
}
with open(OUT / "config.json", "w", encoding="utf-8") as f:
    json.dump(config, f, ensure_ascii=False, indent=2)

print(json.dumps(config, ensure_ascii=False, indent=2))
print(f"\nsalidas en {OUT}")

{
  "experimento": "reranking_dos_etapas",
  "metodologia": "Cuaderno 6, Semana 3 (MCC225), adaptado a datos pequeños",
  "bi_encoder": "openai/clip-vit-large-patch14",
  "variante_imagenes": "con_titulo",
  "caption": "caption_3",
  "topk": 10,
  "negativos": "semi-duros del top-K del bi-encoder",
  "validacion": "dejando una edición fuera (3 pliegues)",
  "clip_congelado": true,
  "epochs": 20,
  "lr": 0.001,
  "seed": 22514,
  "n_total": 94,
  "runtime": {
    "python": "3.11.0rc1",
    "torch": "2.5.1+cu121",
    "gpu": "NVIDIA GeForce RTX 4070 Laptop GPU"
  }
}

salidas en /tf/work/final/sbs_iesf_pares/resultados_reranking_con_titulo
